# BPS Range Bar data

In [ ]:
from ib_async import *

util.startLoop()

ib = IB()
ib.connect("127.0.0.1", 8888, clientId=14)
#ib.reqMarketDataType(1)  # Delayed frozen

Instruct the notebook to draw plot graphics inline:

In [ ]:
%matplotlib inline

Install events handlers

In [ ]:
# handle incoming market data
def onTickerUpdate(tickers):
    for t in tickers:
        pass
def tickfilter_update_tick_bars(ticker):
    df = util.df(ticker)
    print('TICK BARS = \n')
    print(df)
def tickfilter_update_volume_bars(ticker):
    df = util.df(ticker)
    print('VOLUME BARS = \n')
    print(df)
def tickfilter_update_time_bars(ticker):
    df = util.df(ticker)
    print('TIME BARS = \n')
    print(df)
def tickfilter_update_value_bars(ticker):
    df = util.df(ticker)
    print('VALUE BARS = \n')
    print(df)    
def tickfilter_update_rangebars(ticker):
    df = util.df(ticker)
    print('RANGE BARS = \n')
    print(df)    


Request market data for SPY

In [ ]:
# set up contract / request mkt data
# contract = Contract(symbol = 'SPY', secType='STK', currency='USD', exchange='SMART')
contract = Contract(symbol = 'ES', secType='FUT', lastTradeDateOrContractMonth='202609', exchange='CME')
ib.qualifyContracts(contract)
multiplier = int(contract.multiplier)
ticker = ib.reqMktData(contract, "", False, False)
ib.sleep(2)
#ticker = ib.ticker(contract)


In [ ]:
# normal processing of streaming L1 data
ib.pendingTickersEvent += onTickerUpdate
# aggregate tickers by count
tickfilter_tick_bars = ib.ticker(contract).updateEvent.trades().tickbars(5)
#tickfilter_tick_bars += tickfilter_update_tick_bars
# aggregate based on volume
tickfilter_volume_bars = ib.ticker(contract).updateEvent.trades().volumebars(20)
#tickfilter_volume_bars += tickfilter_update_volume_bars
# aggregate based on cost
tickfilter_value_bars = ib.ticker(contract).updateEvent.trades().valuebars(10000000, multiplier)
#tickfilter_value_bars += tickfilter_update_value_bars
# aggregate based on basis point range
tickfilter_range_bars = ib.ticker(contract).updateEvent.trades().rangebars(0.5)
#tickfilter_range_bars += tickfilter_update_rangebars
# aggregate per a triggered event
#tickfilter_time_bars = ib.ticker(contract).updateEvent.trades().timebars(ib.pendingTickersEvent) # i haven't tested the ib.pendingTickersEvent yet. presumably it will work
#tickfilter_time_bars += tickfilter_update_time_bars


Replot for every change of the last bar:

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

db = 0
rb = 0
vb = 0

def onRangeBarUpdate(bars, hasNewBar):
    plt.close()
    # plot = util.barplot(bars)
    plot1 = util.barplot(bars[-200:], title=f"{contract.symbol}-range");    
    clear_output(wait=True)
    display(plot1)
    #clear_plots()
    rb+= 1

def onVolBarUpdate(bars, hasNewBar):
    plt.close()
    # plot = util.barplot(bars)
    plot2 = util.barplot(bars[-200:], title=f"{contract.symbol}-volume");    
    clear_output(wait=True)
    display(plot2)
    #clear_plots()
    vb+= 1

def onValueBarUpdate(bars, hasNewBar):
    plt.close()
    # plot = util.barplot(bars)
    plot3 = util.barplot(bars[-200:], title=f"{contract.symbol}-value");    
    clear_output(wait=True)
    display(plot3)
    #clear_plots()
    db+= 1

def clear_plots():
    if db > 1 and vb > 1 and rb > 1:
        clear_output(wait=True)
        rb = vb = db = 0
    
tickfilter_range_bars.bars.updateEvent.clear()

tickfilter_range_bars.bars.updateEvent += onRangeBarUpdate
#tickfilter_volume_bars.bars.updateEvent += onVolBarUpdate
#tickfilter_value_bars.bars.updateEvent += onValueBarUpdate

# ib.sleep(600)


In [ ]:
ib.cancelMktData(contract)

Disconnect IB

In [ ]:
ib.disconnect()